In [1]:
import pyspark as pyspark
from pyspark.sql import SparkSession


In [2]:
pyspark.__file__

'/home/sultan_ali/spark/spark-3.5.9-bin-hadoop3/python/pyspark/__init__.py'

In [3]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

26/11/24 08:12:36 WARN Utils: Your hostname, LAPTOP-FH4AVPBT resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/24 08:12:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 08:12:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
df = spark.read \
    .option("header","true") \
    .csv('../data/fhvhv_tripdata_2021-01.csv.gz')
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [5]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:33:44', dropoff_datetime='2021-01-01 00:49:07', PULocationID='230', DOLocationID='166', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', pickup_datetime='2021-01-01 00:55:19', dropoff_datetime='2021-01-01 01:18:21', PULocationID='152', DOLocationID='167', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:23:56', dropoff_datetime='2021-01-01 00:38:05', PULocationID='233', DOLocationID='142', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:42:51', dropoff_datetime='2021-01-01 00:45:50', PULocationID='142', DOLocationID='143', SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02764', pickup_datetime='2021-01-01 00:48:14', dropoff_datetime='2021-01-01 01:08:42', PULocationID='143', DOLocationID='78', SR_Flag=None)]

### 1. Fix cols data types

In [6]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', StringType(), True), StructField('dropoff_datetime', StringType(), True), StructField('PULocationID', StringType(), True), StructField('DOLocationID', StringType(), True), StructField('SR_Flag', StringType(), True)])

In [7]:
from pyspark.sql import types

In [8]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True), 
    types.StructField('dispatching_base_num', types.StringType(), True), 
    types.StructField('pickup_datetime', types.TimestampType(), True), 
    types.StructField('dropoff_datetime', types.TimestampType(), True), 
    types.StructField('PULocationID', types.IntegerType(), True), 
    types.StructField('DOLocationID', types.IntegerType(), True), 
    types.StructField('SR_Flag', types.StringType(), True)
])

In [9]:
df = spark.read \
    .option("header","true") \
    .schema(schema) \
    .csv('../data/fhvhv_tripdata_2021-01.csv.gz')
df.show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [10]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('SR_Flag', StringType(), True)])

In [11]:
!ls

Untitled.ipynb	fhvhv


In [12]:
df = df.repartition(24)

In [13]:
df.write.parquet('fhvhv/2021/01')

AnalysisException: [PATH_ALREADY_EXISTS] Path file:/home/sultan_ali/data-engineering-zoomcamp-Spark/02_/fhvhv/2021/01 already exists. Set mode as "overwrite" to overwrite the existing path.

In [14]:
df_parq = spark.read.parquet('fhvhv/2021/01')

In [15]:
df_parq

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [16]:
df_parq.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [17]:
display(df_parq.head(5))

[Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 10, 17, 40, 1), dropoff_datetime=datetime.datetime(2021, 1, 10, 17, 48, 10), PULocationID=97, DOLocationID=25, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 8, 18, 19, 44), dropoff_datetime=datetime.datetime(2021, 1, 8, 18, 55, 57), PULocationID=138, DOLocationID=265, SR_Flag=None),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02876', pickup_datetime=datetime.datetime(2021, 1, 1, 16, 47, 20), dropoff_datetime=datetime.datetime(2021, 1, 1, 16, 58, 28), PULocationID=50, DOLocationID=163, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pickup_datetime=datetime.datetime(2021, 1, 15, 17, 50, 8), dropoff_datetime=datetime.datetime(2021, 1, 15, 18, 7, 24), PULocationID=163, DOLocationID=79, SR_Flag=None),
 Row(hvfhs_license_num='HV0005', dispatching_base_num='B02510', pic

In [18]:
df_parq.columns

['hvfhs_license_num',
 'dispatching_base_num',
 'pickup_datetime',
 'dropoff_datetime',
 'PULocationID',
 'DOLocationID',
 'SR_Flag']

In [19]:
df_parq.count()

11908468

In [20]:
df_parq

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

###  2. Spark Df Basics

####  Select colums

In [21]:
df_parq.select("hvfhs_license_num", "DOLocationID") \
    .show(3)

+-----------------+------------+
|hvfhs_license_num|DOLocationID|
+-----------------+------------+
|           HV0005|          25|
|           HV0005|         265|
|           HV0003|         163|
+-----------------+------------+
only showing top 3 rows



In [22]:
df_parq.select("hvfhs_license_num", "DOLocationID") \
    .schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('DOLocationID', IntegerType(), True)])

####  Select/Filter Rows

In [23]:
df_parq \
    .filter(
        (df_parq.DOLocationID <= 100) &
        (df_parq.DOLocationID >= 90)
    ) \
    .show()

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02765|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|   NULL|
|           HV0005|              B02510|2021-01-16 00:28:56|2021-01-16 01:27:38|          92|          92|   NULL|
|           HV0003|              B02765|2021-01-16 15:59:36|2021-01-16 16:26:51|         113|          97|   NULL|
|           HV0003|              B02884|2021-01-13 17:30:24|2021-01-13 17:50:02|         241|          94|   NULL|
|           HV0005|              B02510|2021-01-16 20:13:09|2021-01-16 20:31:14|         210|          91|   NULL|
|           HV0003|              B02765|2021-01-24 21:45:57|2021-01-24 21:49:53|

In [24]:
df_parq \
    .select("hvfhs_license_num", "dispatching_base_num", "DOLocationID") \
    .filter(
        (df_parq.DOLocationID <= 100) &
        (df_parq.DOLocationID >= 90)
    ) \
    .show()

+-----------------+--------------------+------------+
|hvfhs_license_num|dispatching_base_num|DOLocationID|
+-----------------+--------------------+------------+
|           HV0003|              B02765|          95|
|           HV0005|              B02510|          92|
|           HV0003|              B02765|          97|
|           HV0003|              B02884|          94|
|           HV0005|              B02510|          91|
|           HV0003|              B02765|          96|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|          95|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|         100|
|           HV0003|              B02882|          90|
|           HV0003|              B02888|          92|
|           HV0003|              B02875|          90|
|           HV0003|              B02875|          91|
|           HV0005|              B02510|          97|
|           HV0003|         

In [29]:
df_parq

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, pickup_datetime: timestamp, dropoff_datetime: timestamp, PULocationID: int, DOLocationID: int, SR_Flag: string]

In [31]:
df_parq.select("hvfhs_license_num") \
    .distinct() \
    .show()

+-----------------+
|hvfhs_license_num|
+-----------------+
|           HV0004|
|           HV0005|
|           HV0003|
+-----------------+



In [38]:

df_parq.withColumn(
        "DOLocationID+PULocationID",
        col("DOLocationID")
    ) \
    .select("hvfhs_license_num", "dispatching_base_num", "DOLocationID", "DOLocationID+PULocationID") \
    .show()

+-----------------+--------------------+------------+-------------------------+
|hvfhs_license_num|dispatching_base_num|DOLocationID|DOLocationID+PULocationID|
+-----------------+--------------------+------------+-------------------------+
|           HV0005|              B02510|          25|                       25|
|           HV0005|              B02510|         265|                      265|
|           HV0003|              B02876|         163|                      163|
|           HV0005|              B02510|          79|                       79|
|           HV0005|              B02510|          74|                       74|
|           HV0003|              B02765|          95|                       95|
|           HV0003|              B02872|          77|                       77|
|           HV0005|              B02510|          41|                       41|
|           HV0003|              B02764|         244|                      244|
|           HV0003|              B02764|

In [40]:
df_parq.withColumn(
        "DOLocationID+PULocationID",
        F.concat_ws("_", F.col("DOLocationID"), F.col("PULocationID"))
    ) \
    .select("hvfhs_license_num", "DOLocationID", "PULocationID", "DOLocationID+PULocationID") \
    .show()

+-----------------+------------+------------+-------------------------+
|hvfhs_license_num|DOLocationID|PULocationID|DOLocationID+PULocationID|
+-----------------+------------+------------+-------------------------+
|           HV0005|          25|          97|                    25_97|
|           HV0005|         265|         138|                  265_138|
|           HV0003|         163|          50|                   163_50|
|           HV0005|          79|         163|                   79_163|
|           HV0005|          74|          47|                    74_47|
|           HV0003|          95|          48|                    95_48|
|           HV0003|          77|          63|                    77_63|
|           HV0005|          41|         238|                   41_238|
|           HV0003|         244|          63|                   244_63|
|           HV0003|         165|         210|                  165_210|
|           HV0003|         143|         113|                  1

### 3. Func available inSpark

In [43]:
from pyspark.sql import functions as F

In [44]:
df_parq \
    .select("hvfhs_license_num", "dispatching_base_num", "DOLocationID") \
    .filter(
        (F.col("DOLocationID") <= 100) &
        (F.col("DOLocationID") >= 90)
    ) \
    .show()

# df_parq.DOLocationID = col("DOLocationID")

+-----------------+--------------------+------------+
|hvfhs_license_num|dispatching_base_num|DOLocationID|
+-----------------+--------------------+------------+
|           HV0003|              B02765|          95|
|           HV0005|              B02510|          92|
|           HV0003|              B02765|          97|
|           HV0003|              B02884|          94|
|           HV0005|              B02510|          91|
|           HV0003|              B02765|          96|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|          95|
|           HV0003|              B02835|          92|
|           HV0005|              B02510|         100|
|           HV0003|              B02882|          90|
|           HV0003|              B02888|          92|
|           HV0003|              B02875|          90|
|           HV0003|              B02875|          91|
|           HV0005|              B02510|          97|
|           HV0003|         

In [46]:
df_parq.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [47]:
df_parq. \
    withColumn(
        'pickup_date',
        F.to_date(df_parq.pickup_datetime)
    ). \
    select(
        "hvfhs_license_num",
        "pickup_datetime",
        "pickup_date"
    ).show()

+-----------------+-------------------+-----------+
|hvfhs_license_num|    pickup_datetime|pickup_date|
+-----------------+-------------------+-----------+
|           HV0005|2021-01-10 17:40:01| 2021-01-10|
|           HV0005|2021-01-08 18:19:44| 2021-01-08|
|           HV0003|2021-01-01 16:47:20| 2021-01-01|
|           HV0005|2021-01-15 17:50:08| 2021-01-15|
|           HV0005|2021-01-12 17:26:40| 2021-01-12|
|           HV0003|2021-01-05 02:00:14| 2021-01-05|
|           HV0003|2021-01-02 00:34:43| 2021-01-02|
|           HV0005|2021-01-06 23:24:42| 2021-01-06|
|           HV0003|2021-01-02 16:20:11| 2021-01-02|
|           HV0003|2021-01-24 16:00:53| 2021-01-24|
|           HV0003|2021-01-16 19:35:17| 2021-01-16|
|           HV0005|2021-01-28 22:17:58| 2021-01-28|
|           HV0003|2021-01-01 11:15:17| 2021-01-01|
|           HV0003|2021-01-19 12:05:32| 2021-01-19|
|           HV0003|2021-01-17 13:54:52| 2021-01-17|
|           HV0003|2021-01-30 18:03:33| 2021-01-30|
|           

### 4. User define func

In [60]:
from datetime import datetime

def trip_weirdness_score(
    license_num: str | None,
    pickup: datetime | None,
    dropoff: datetime | None,
) -> float | None:
    
    LICENSE_MULTIPLIER = {
        "HV0004": 1.2,
        "HV0005": 0.9,
        "HV0003": 1.05,
    }
    if pickup is None or dropoff is None:
        return None

    duration_sec = (dropoff - pickup).total_seconds()
    score = 0.0

    # 1. Negative or ultra-short trips (e.g., < 15s) or multi-hour outliers (> 6h)
    if duration_sec < 0:
        score += 100.0
    elif duration_sec < 15:
        score += 25.0
    elif duration_sec > 6 * 3600:
        score += 30.0
        
    mult = LICENSE_MULTIPLIER.get(license_num or "", 1.0)
    return round(score * mult, 2)

In [61]:
# Here is user difiene fuction created
weirdness_udf = F.udf(trip_weirdness_score, types.DoubleType())

In [62]:
df_scored = (
    df_parq.withColumn(
        "trip_weirdness_score",
        weirdness_udf(
            F.col("hvfhs_license_num"),
            F.col("pickup_datetime"),
            F.col("dropoff_datetime")
        ),
    )
    .filter(F.col("trip_weirdness_score") >= 25.0)
    .select(
        "hvfhs_license_num",
        "dispatching_base_num",
        "PULocationID",
        "DOLocationID",
        "SR_Flag",
        "trip_weirdness_score",
    )
)

df_scored.show(20, truncate=False)

+-----------------+--------------------+------------+------------+-------+--------------------+
|hvfhs_license_num|dispatching_base_num|PULocationID|DOLocationID|SR_Flag|trip_weirdness_score|
+-----------------+--------------------+------------+------------+-------+--------------------+
|HV0003           |B02617              |254         |254         |NULL   |26.25               |
|HV0004           |B02800              |78          |213         |NULL   |30.0                |
|HV0004           |B02800              |166         |166         |NULL   |30.0                |
|HV0003           |B02869              |227         |265         |NULL   |31.5                |
|HV0003           |B02864              |234         |234         |NULL   |26.25               |
|HV0004           |B02800              |151         |237         |NULL   |30.0                |
|HV0003           |B02765              |90          |90          |NULL   |26.25               |
|HV0004           |B02800              |